#  Tầng 2b: Multiple Answer Cleanup

**Mục tiêu:** Nhận diện các cột Checkbox (nhiều đáp án phân cách bằng dấu phẩy),  
áp dụng One-Hot Encoding để tách thành các cột binary (0/1).

**Input:** `cleaned_responses.csv`  
**Output:** `multiple_answers_processed.csv`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Config ──────────────────────────────────────────────
INPUT_FILE = "cleaned_responses.csv"
OUTPUT_FILE = "multiple_answers_processed.csv"

assert Path(INPUT_FILE).exists(), f" Chưa có {INPUT_FILE}. Hãy chạy 1a_cleanup.ipynb trước!"
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
print(f" Đọc {INPUT_FILE}: {df.shape[0]} dòng × {df.shape[1]} cột")

✅ Đọc cleaned_responses.csv: 151 dòng × 41 cột


In [ ]:
# ══════════════════════════════════════════════════════════
# BƯỚC 1: TỰ ĐỘNG NHẬN DIỆN CỘT CHECKBOX
# ══════════════════════════════════════════════════════════
# Logic: Cột checkbox là cột object có ít nhất 1 giá trị chứa dấu phẩy
# và KHÔNG phải cột free-text (loại trừ Q35, Q36, Q37)

FREE_TEXT_COLS = ["Q35_improvement_suggestions", "Q36_university_suggestions", "Q37_skills_needed"]

def detect_checkbox_columns(dataframe, exclude_cols=None):
    """Tự động nhận diện cột checkbox dựa trên dấu phẩy trong giá trị."""
    exclude_cols = exclude_cols or []
    obj_cols = dataframe.select_dtypes(include="object").columns
    
    checkbox_cols = []
    for col in obj_cols:
        if col in exclude_cols or col == "timestamp":
            continue
        # Đếm số dòng có chứa dấu phẩy (loại bỏ NaN)
        has_comma = dataframe[col].dropna().str.contains(",", regex=False)
        comma_ratio = has_comma.mean()
        if comma_ratio > 0.1:  # > 10% dòng có dấu phẩy → checkbox
            checkbox_cols.append(col)
    
    return checkbox_cols

CHECKBOX_COLS = detect_checkbox_columns(df, exclude_cols=FREE_TEXT_COLS)

print(f" Phát hiện {len(CHECKBOX_COLS)} cột Checkbox:")
for col in CHECKBOX_COLS:
    n_multi = df[col].str.contains(",", na=False).sum()
    print(f"   • {col}  ({n_multi}/{len(df)} dòng có nhiều đáp án)")

🔍 Phát hiện 8 cột Checkbox:
   • Q4_product_field  (103/151 dòng có nhiều đáp án)
   • Q7_tools_used  (18/151 dòng có nhiều đáp án)
   • Q9_usage_purpose  (42/151 dòng có nhiều đáp án)
   • Q10_cicd_benefits  (60/151 dòng có nhiều đáp án)
   • Q33_cicd_difficulties  (60/151 dòng có nhiều đáp án)
   • Q34_adoption_barriers  (72/151 dòng có nhiều đáp án)
   • Q39_biggest_barrier  (23/151 dòng có nhiều đáp án)
   • Q40_expected_benefit  (38/151 dòng có nhiều đáp án)


In [ ]:
# ══════════════════════════════════════════════════════════
# BƯỚC 2: ONE-HOT ENCODING cho từng cột Checkbox
# ══════════════════════════════════════════════════════════

def one_hot_encode_checkbox(series, col_prefix):
    """
    Chuyển cột checkbox (giá trị phân cách bằng dấu phẩy) → One-Hot DataFrame.
    Sử dụng vectorized operations thay vì vòng lặp for.
    
    Parameters:
        series: pd.Series chứa các giá trị checkbox
        col_prefix: tiền tố cho tên cột output
    
    Returns:
        pd.DataFrame với các cột binary (0/1)
    """
    # Split mỗi giá trị thành list, strip whitespace
    split_series = series.str.split(",").apply(
        lambda x: [item.strip() for item in x] if isinstance(x, list) else []
    )
    
    # Dùng MultiLabelBinarizer logic bằng pandas
    # Tạo DataFrame từ explode + get_dummies
    exploded = split_series.explode()
    
    # Loại bỏ giá trị rỗng và "Không trả lời"
    exploded = exploded[exploded.str.len() > 0]
    exploded = exploded[exploded != "Không trả lời"]
    
    if exploded.empty:
        return pd.DataFrame(index=series.index)
    
    # One-hot via crosstab
    dummies = pd.crosstab(exploded.index, exploded)
    
    # Clip to binary (0 or 1)
    dummies = dummies.clip(upper=1)
    
    # Reindex to match original index (fill missing rows with 0)
    dummies = dummies.reindex(series.index, fill_value=0)
    
    # Rename columns with prefix
    dummies.columns = [f"{col_prefix}__{c}" for c in dummies.columns]
    
    return dummies

print(" Hàm one_hot_encode_checkbox đã sẵn sàng")

✅ Hàm one_hot_encode_checkbox đã sẵn sàng


In [ ]:
# ══════════════════════════════════════════════════════════
# BƯỚC 3: ÁP DỤNG ONE-HOT cho tất cả cột Checkbox
# ══════════════════════════════════════════════════════════

onehot_frames = []

for col in CHECKBOX_COLS:
    ohe = one_hot_encode_checkbox(df[col], col)
    onehot_frames.append(ohe)
    print(f"   {col} → {ohe.shape[1]} dummy columns")
    # Preview unique values
    short_names = [c.replace(f"{col}__", "")[:50] for c in ohe.columns]
    for name in short_names:
        print(f"      └─ {name}")
    print()

# Ghép tất cả dummy columns vào df gốc
df_processed = pd.concat([df] + onehot_frames, axis=1)

print(f"\n Shape sau One-Hot: {df_processed.shape[0]} dòng × {df_processed.shape[1]} cột")
print(f"   (Thêm {df_processed.shape[1] - df.shape[1]} cột dummy)")

  ✅ Q4_product_field → 14 dummy columns
      └─ Bay
      └─ Chưa có
      └─ Desktop App)
      └─ Game
      └─ Giải pháp hạ tầng & Bảo mật
      └─ Hệ thống AI/Big Data/Data Science
      └─ Hệ thống nhúng (Embedded)/IoT
      └─ Không
      └─ Không có
      └─ Ko
      └─ Mobile
      └─ Network
      └─ Nhi béo
      └─ Phần mềm ứng dụng (Web

  ✅ Q7_tools_used → 7 dummy columns
      └─ Chưa từng sử dụng công cụ CI/CD nào
      └─ Circle CI
      └─ GitHub Actions
      └─ GitLab CI/CD
      └─ Jenkins
      └─ Nhi béo
      └─ Travis CI

  ✅ Q9_usage_purpose → 8 dummy columns
      └─ Automated Testing – kiểm thử tự động
      └─ Build / Tự động hoá quá trình build
      └─ Chưa từng sử dụng
      └─ Continuous Deployment / Delivery (CD) – triển khai
      └─ Continuous Integration (CI) – tích hợp & kiểm tra 
      └─ Học tập / nghiên cứu / thử nghiệm cá nhân
      └─ Nhi béo
      └─ Tự động hoá quy trình phát triển (workflow automat

  ✅ Q10_cicd_benefits → 6 dummy columns
 

In [ ]:
# ══════════════════════════════════════════════════════════
# BƯỚC 4: THỐNG KÊ TẦN SUẤT cho mỗi cột Checkbox
# ══════════════════════════════════════════════════════════

for col in CHECKBOX_COLS:
    dummy_cols = [c for c in df_processed.columns if c.startswith(f"{col}__")]
    if dummy_cols:
        freq = df_processed[dummy_cols].sum().sort_values(ascending=False)
        freq.index = [c.replace(f"{col}__", "") for c in freq.index]
        print(f"\n {col} - Tần suất:")
        print(freq.to_string())
        print(f"   Total responses: {len(df_processed)}")


📊 Q4_product_field - Tần suất:
Desktop App)                         103
Phần mềm ứng dụng (Web               103
Mobile                               103
Hệ thống AI/Big Data/Data Science     29
Hệ thống nhúng (Embedded)/IoT          6
Giải pháp hạ tầng & Bảo mật            4
Chưa có                                2
Bay                                    1
Game                                   1
Không                                  1
Ko                                     1
Không có                               1
Network                                1
Nhi béo                                1
   Total responses: 151

📊 Q7_tools_used - Tần suất:
GitHub Actions                         47
Chưa từng sử dụng công cụ CI/CD nào    38
GitLab CI/CD                           23
Jenkins                                10
Circle CI                               7
Travis CI                               5
Nhi béo                                 1
   Total responses: 151

📊 Q9_usage_purpose - T

In [ ]:
# ══════════════════════════════════════════════════════════
# BƯỚC 5: LƯU FILE
# ══════════════════════════════════════════════════════════

df_processed.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# Verify
verify = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
assert verify.shape == df_processed.shape
print(f" Đã lưu: {OUTPUT_FILE}")
print(f"   Shape: {verify.shape[0]} dòng × {verify.shape[1]} cột")
print(f"   Trong đó: {len(df.columns)} cột gốc + {verify.shape[1] - len(df.columns)} cột One-Hot")

✅ Đã lưu: multiple_answers_processed.csv
   Shape: 151 dòng × 114 cột
   Trong đó: 41 cột gốc + 73 cột One-Hot
